# SG Citizen Financial Assistant - Colab Demo
Clones the repo, installs dependencies, and runs the FastAPI backend + static frontend with a public tunnel for live demos.

**The repo does not carry any data.** `.gitignore` excludes `data/raw/`, `data/processed/` and `data/faiss/`, so a fresh clone has neither the source corpus nor a prebuilt FAISS index, and the backend will answer `503 Knowledge base index not found` until one is supplied. Pick one of the two options below before starting the server:

- **Option A (fast, recommended):** copy a FAISS index you built locally (`data/faiss/index.faiss` + `data/faiss/metadata.jsonl`) into Google Drive and mount it - no GPU time, no re-embedding.
- **Option B:** copy the raw corpus (`data/raw/`) into Drive and re-run ingestion in Colab. Slower, and it needs Tesseract for infographics plus a Gemini key for video transcripts.


In [ ]:
!git clone https://github.com/YOUR_ORG/sg-citizen-financial-assistant.git
%cd sg-citizen-financial-assistant
!pip install -q -r requirements.txt pyngrok


In [ ]:
# Option A: mount Drive and copy a prebuilt index into data/faiss/
from google.colab import drive
drive.mount('/content/drive')

# Adjust to wherever you uploaded index.faiss + metadata.jsonl
DRIVE_INDEX_DIR = '/content/drive/MyDrive/sg-financial-assistant/faiss'

!mkdir -p data/faiss
!cp "$DRIVE_INDEX_DIR"/index.faiss data/faiss/index.faiss
!cp "$DRIVE_INDEX_DIR"/metadata.jsonl data/faiss/metadata.jsonl
!ls -l data/faiss


In [ ]:
# Option B (skip if you ran Option A): copy the raw corpus from Drive and rebuild the index here.
# Needs Tesseract for infographic OCR; video transcripts also need GEMINI_API_KEY set (next cell).
# from google.colab import drive
# drive.mount('/content/drive')
# !apt-get -qq install -y tesseract-ocr
# !mkdir -p data/raw && cp -r /content/drive/MyDrive/sg-financial-assistant/raw/. data/raw/
# !python -m ingestion.build_index


In [ ]:
from google.colab import userdata
import os
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
os.environ['GROK_API_KEY'] = userdata.get('GROK_API_KEY') if userdata.get('GROK_API_KEY') else ''
os.environ['LLM_PROVIDER'] = 'gemini'

In [ ]:
# Sanity check: the index must exist before the server can answer anything.
from pathlib import Path

for path in (Path('data/faiss/index.faiss'), Path('data/faiss/metadata.jsonl')):
    assert path.exists(), f'Missing {path} - run Option A or Option B above first'
print('Index present:', sum(1 for _ in open('data/faiss/metadata.jsonl', encoding='utf-8')), 'chunks')


In [ ]:
import subprocess
from pyngrok import ngrok

server = subprocess.Popen(['uvicorn', 'backend.main:app', '--host', '0.0.0.0', '--port', '8000'])
public_url = ngrok.connect(8000)
print('Public demo URL:', public_url)

In [ ]:
# Run at the end of the demo to stop the server and tunnel
server.terminate()
ngrok.disconnect(public_url)